# 技能4 · Day 5 上机：商业模式画布 + 投资评估

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 核心任务

为 AI 营销 Agent SaaS「MarketingAgent Pro」构建完整投资评估：
1. 商业模式画布（9宫格结构化）
2. DCF 估值（NPV / IRR / 回收期 / PI）
3. 蒙特卡洛模拟（估值分布 + 概率分析）
4. 敏感性分析（龙卷风图）
5. 天道推演多路径场景分析

**真实库**：numpy-financial（NPV/IRR）｜ scipy.stats（蒙特卡洛）｜ pandas + matplotlib
**真实数据**：HubSpot 2023 财报 + Jasper AI Crunchbase + 独立教材 MarketingAgent Pro 单位经济模型


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：

> 需要 numpy-financial, scipy, pandas, matplotlib。通常已随 conda/venv 安装。
> numpy-financial 提供 NPV/IRR 等标准金融函数。


In [ ]:
# !pip install numpy-financial scipy pandas matplotlib -q
import numpy as np
import pandas as pd
import numpy_financial as npf
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("环境就绪: numpy-financial + scipy.stats + pandas + matplotlib")


## 1. 商业模式画布（Business Model Canvas）

AI 商业模式画布在传统九宫格基础上适配 AI 原生特征：
- **收入流**：新增 outcome-based pricing + Agent 交易费
- **核心资源**：新增数据资产 + AI 模型 + 算力
- **核心活动**：新增模型训练/评估 + Agent 运维
- **成本结构**：新增推理成本（持续运营成本）

**案例**：MarketingAgent Pro - AI 原生营销 Agent 平台
- 数据校准：HubSpot 2023 财报（gross margin ~78%）、Jasper AI（$125M ARR）、独立教材 Day 5 单位经济模型


In [ ]:
# 1. 商业模式画布（9宫格）
# 数据校准: HubSpot 2023 gross margin ~78%, Jasper AI $125M ARR, 独立教材Day5
canvas_data = {
    '构件': ['客户细分', '价值主张', '渠道', '客户关系', '收入流',
            '核心资源', '核心活动', '核心伙伴', '成本结构'],
    'MarketingAgent Pro': [
        '中型企业营销部门(50-500人, 月预算10-100万)',
        'AI Agent替代50%营销重复性工作, 营销人聚焦策略和创意',
        '直销(大客户) + 自助注册(中小) + Agent市场分发',
        'AI驱动个性化onboarding + Customer Success Agent',
        '基础订阅$500/月 + 按结果付费$5-50/转化 + 企业定制',
        '营销专有数据(脱敏campaign) + Agent编排 + 行业Know-how',
        'Agent训练/优化 + 数据管道运维 + 效果评估 + 客户成功',
        '基础模型(OpenAI/Anthropic) + 广告平台API + 数据伙伴',
        '推理成本30% + 数据15% + 人才30% + 营销15% + 合规10%'
    ],
    '传统SaaS对比': [
        '同上(AI未改变目标客户)',
        '工具辅助(人为主, AI为辅)',
        '直销+自助注册(无Agent渠道)',
        '人工CSM + 工单系统',
        'Seat-based($X/用户/月)',
        '软件代码 + 客户名单 + 品牌',
        '软件开发 + 客户成功 + 销售',
        '云服务 + 支付 + CDN',
        '研发40% + 销售30% + 运营20% + 合规10%'
    ]
}
canvas_df = pd.DataFrame(canvas_data)
print("=== MarketingAgent Pro 商业模式画布 ===")
print(canvas_df.to_string(index=False))


## 2. DCF 估值模型

DCF（Discounted Cash Flow）是投资评估的核心方法。对 AI SaaS：

| 参数 | 值 | 来源 |
|------|-----|------|
| 初始投资 | $2,000K | 开发团队 + GTM 投入 |
| ARPU | $24K/年 ($2K/月) | 独立教材 Day 5 |
| 毛利率 | 65% | 含推理成本 30% + 数据 5% |
| 折现率 | 15% | VC 典型 SaaS 要求回报 |
| 评估窗口 | 5年 | J 曲线效应需 3-5 年 |

**numpy-financial 核心函数**：
- `npf.npv(rate, cashflows)` — 净现值
- `npf.irr(cashflows)` — 内部收益率


In [ ]:
# 2. DCF 5年财务模型 + NPV
# 基准: HubSpot gross margin ~78%, AI SaaS含推理成本降至65%
initial_investment = 2000  # $K
years = list(range(6))
customers = [0, 30, 80, 160, 260, 380]
arpu_annual = 24  # $K/年 ($2K/月 x 12)
revenue = [c * arpu_annual for c in customers]
gross_margin = 0.65
gross_profit = [r * gross_margin for r in revenue]
opex = [0, 800, 1200, 1800, 2500, 3200]
fcf = [gp - op for gp, op in zip(gross_profit, opex)]
fcf[0] = -initial_investment
discount_rate = 0.15

dcf_df = pd.DataFrame({
    'Year': years,
    'Customers': customers,
    'Revenue($K)': [r for r in revenue],
    'GrossProfit($K)': [round(gp, 1) for gp in gross_profit],
    'OpEx($K)': opex,
    'FCF($K)': [round(f, 1) for f in fcf]
})
print("=== DCF 5年财务模型 ===")
print(dcf_df.to_string(index=False))

npv = npf.npv(discount_rate, fcf)
print(f"\nNPV @ {discount_rate:.0%} = ${npv:.1f}K")


In [ ]:
# 3. IRR + 回收期 + 盈利指数
irr = npf.irr(fcf)

# 回收期(手动计算, numpy_financial无此函数)
cumulative = 0
payback_period = None
for i, cf in enumerate(fcf):
    cumulative += cf
    if cumulative >= 0 and i > 0:
        prev_cum = cumulative - cf
        payback_period = (i - 1) + (-prev_cum) / cf
        break

# 盈利指数 PI = PV(未来现金流) / |初始投资|
pv_future = sum(fcf[i] / (1 + discount_rate)**i for i in range(1, len(fcf)))
pi = pv_future / abs(fcf[0])

print(f"IRR: {irr:.2%}")
print(f"Payback Period: {payback_period:.1f} 年" if payback_period else "Payback Period: >5年")
print(f"Profitability Index: {pi:.2f}")
print(f"投资可行: IRR>{discount_rate:.0%}? {'是' if irr > discount_rate else '否'} | PI>1? {'是' if pi > 1 else '否'}")


## 3. 蒙特卡洛模拟（Monte Carlo Simulation）

DCF 给出 NPV 的点估计，但 AI SaaS 的关键参数高度不确定：
- **推理成本**：模型 API 价格快速变化（GPT-4 -> DeepSeek 成本降 90%+）
- **ARPU**：outcome-based pricing 下波动大
- **客户增长**：市场竞争 + 产品成熟度不确定
- **毛利率**：推理成本曲线决定长期毛利

蒙特卡洛方法：对不确定参数抽样 -> 计算每次抽样的 NPV -> 得到估值分布

**scipy.stats / numpy 分布**：
- `np.random.normal(mu, sigma, n)` — 正态分布抽样
- `np.clip(arr, low, high)` — 截断分布范围
- `np.percentile(arr, q)` — 分位数


In [ ]:
# 4. 蒙特卡洛模拟 (10000次)
n_sim = 10000

# 参数分布 (基于真实AI SaaS行业基准)
# ARPU: $2K/月 ± $3K/年 (独立教材基准)
# Gross margin: 65% ± 5% (HubSpot ~78%, AI推理成本拉低)
# Growth multiplier: 1.0 ± 0.2 (客户增长不确定性)
# OpEx multiplier: 1.0 ± 0.15 (运营成本不确定性)
arpus_sim = np.random.normal(24, 3, n_sim)
margins_sim = np.clip(np.random.normal(0.65, 0.05, n_sim), 0.35, 0.85)
growth_mults = np.clip(np.random.normal(1.0, 0.2, n_sim), 0.5, 1.5)
opex_mults = np.random.normal(1.0, 0.15, n_sim)

npv_sim = np.zeros(n_sim)
for i in range(n_sim):
    # 缩放base case
    cust_i = [max(int(c * growth_mults[i]), 1) for c in customers]
    rev_i = [c * arpus_sim[i] for c in cust_i]
    gp_i = [r * margins_sim[i] for r in rev_i]
    op_i = [o * opex_mults[i] for o in opex]
    cf_i = [gp_i[j] - op_i[j] for j in range(6)]
    cf_i[0] = -initial_investment
    npv_sim[i] = npf.npv(discount_rate, cf_i)

print(f"=== Monte Carlo 估值分布 (n={n_sim}) ===")
print(f"  均值 (Mean NPV):   ${npv_sim.mean():.1f}K")
print(f"  中位数 (Median):   ${np.median(npv_sim):.1f}K")
print(f"  标准差 (Std):      ${npv_sim.std():.1f}K")
print(f"  5%分位 (P5):       ${np.percentile(npv_sim, 5):.1f}K")
print(f"  95%分位 (P95):     ${np.percentile(npv_sim, 95):.1f}K")
print(f"  P(NPV > 0):        {(npv_sim > 0).mean():.1%}")


## 4. 敏感性分析（龙卷风图）

龙卷风图（Tornado Chart）展示各参数对 NPV 的影响排序：
- 对每个参数 ±20% 变动，计算 NPV 变化范围
- 按影响大小降序排列，形成龙卷风形状
- 识别**高杠杆点**：小投入改变大局的关键参数

**2026前沿 — 推理成本对 AI 估值的影响**：
推理成本是 AI SaaS 估值的核心变量。DeepSeek 等开源模型将推理成本降低 90%+，
直接提升毛利率和估值。敏感性分析帮助量化这一影响。


In [ ]:
# 5. 敏感性分析 + 龙卷风图
def calc_npv(arpu=24, inference_ratio=0.30, data_ratio=0.05, growth_mult=1.0, opex_mult=1.0, dr=0.15):
    """计算给定参数下的NPV ($K)
    margin = 1 - inference_ratio - data_ratio
    """
    margin = 1.0 - inference_ratio - data_ratio
    cust = [max(int(c * growth_mult), 1) for c in customers]
    rev = [c * arpu for c in cust]
    gp = [r * margin for r in rev]
    op = [o * opex_mult for o in opex]
    cf = [gp[j] - op[j] for j in range(6)]
    cf[0] = -initial_investment
    return npf.npv(dr, cf)

base_npv = calc_npv()

# 各参数±20%变动 (推理成本下降->毛利率上升->NPV上升)
params_test = {
    'ARPU':            lambda d: calc_npv(arpu=24*(1+d)),
    'Inference Cost':  lambda d: calc_npv(inference_ratio=0.30*(1+d)),
    'Growth':          lambda d: calc_npv(growth_mult=1.0+d),
    'OpEx':            lambda d: calc_npv(opex_mult=1.0+d),
    'Discount Rate':   lambda d: calc_npv(dr=0.15*(1+d)),
}

sensitivity = []
for name, fn in params_test.items():
    npv_high = fn(0.20)
    npv_low = fn(-0.20)
    sensitivity.append({
        '参数': name,
        'Low(-20%)': round(npv_low, 1),
        'Base': round(base_npv, 1),
        'High(+20%)': round(npv_high, 1),
        'Impact': round(abs(npv_high - npv_low), 1)
    })

sensitivity_df = pd.DataFrame(sensitivity).sort_values('Impact', ascending=False)
print("=== 敏感性分析（按影响降序）===")
print(sensitivity_df.to_string(index=False))
print(f"\n最敏感因子: {sensitivity_df.iloc[0]['参数']} (影响: ${sensitivity_df.iloc[0]['Impact']:.1f}K)")

# 龙卷风图
fig, ax = plt.subplots(figsize=(10, 5))
labels = sensitivity_df['参数'].values
lows = sensitivity_df['Low(-20%)'].values
highs = sensitivity_df['High(+20%)'].values
base_val = sensitivity_df['Base'].values[0]
y_pos = range(len(sensitivity_df))
for idx in range(len(sensitivity_df)):
    ax.barh(idx, highs[idx] - base_val, left=base_val, height=0.6, color='steelblue', alpha=0.7)
    ax.barh(idx, lows[idx] - base_val, left=base_val, height=0.6, color='coral', alpha=0.7)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels)
ax.set_xlabel('NPV ($K)')
ax.set_title('敏感性分析 - 龙卷风图 (Tornado Chart)')
ax.axvline(x=base_val, color='black', linestyle='--', label=f'Base NPV=${base_val:.0f}K')
ax.legend()
plt.tight_layout()
plt.savefig('tornado_chart.png', dpi=100, bbox_inches='tight')
plt.show()
print("龙卷风图已保存: tornado_chart.png")


## 5. 天道推演 × 投资评估（2026前沿）

> 与项目 CLAUDE.md「天道推演系统」同构。

天道推演是一种元认知沙盘推演能力——以天神视角俯视局势，构建无限可能的沙盘，
模拟不同决策路径下的未来走向。应用于投资评估：

| 天道推演能力 | 投资评估对应 | 实现方式 |
|-------------|------------|---------|
| 局势感知 | 市场环境建模 | 场景定义 |
| 因果链追踪 | 价值驱动因素分析 | 敏感性分析 |
| 沙盘模拟（3层） | 多路径推演 | Bull / Base / Bear |
| 概率评估 | 估值概率分布 | 蒙特卡洛模拟 |
| 最优路径推荐 | 投资决策 | NPV / IRR / PI |

**三路径推演**：Bull（乐观）/ Base（基准）/ Bear（悲观），每路径推演 3 层（immediate / near / far）。


In [ ]:
# 6. 天道推演多路径场景分析
# 天道推演: 沙盘模拟 -> 多路径推演 -> 概率评估 -> 最优路径
scenarios = {
    'Bull (乐观)': {'arpu': 28, 'inference_ratio': 0.23, 'growth_mult': 1.3, 'opex_mult': 0.9, 'dr': 0.12},
    'Base (基准)': {'arpu': 24, 'inference_ratio': 0.30, 'growth_mult': 1.0, 'opex_mult': 1.0, 'dr': 0.15},
    'Bear (悲观)': {'arpu': 20, 'inference_ratio': 0.40, 'growth_mult': 0.7, 'opex_mult': 1.2, 'dr': 0.20},
}

scenario_results = []
for name, p in scenarios.items():
    npv_s = calc_npv(**p)

    # 3层推演: immediate(Year1-2) / near(Year3-4) / far(Year5)
    margin = 1.0 - p['inference_ratio'] - 0.05  # data_ratio=0.05
    cust = [max(int(c * p['growth_mult']), 1) for c in customers]
    rev = [c * p['arpu'] for c in cust]
    gp = [r * margin for r in rev]
    op = [o * p['opex_mult'] for o in opex]
    cf = [gp[j] - op[j] for j in range(6)]
    cf[0] = -initial_investment

    dr = p['dr']
    pv_imm = cf[1]/(1+dr) + cf[2]/(1+dr)**2
    pv_near = cf[3]/(1+dr)**3 + cf[4]/(1+dr)**4
    pv_far = cf[5]/(1+dr)**5

    scenario_results.append({
        '场景': name,
        'NPV($K)': round(npv_s, 1),
        'Immediate(Y1-2)': round(pv_imm, 1),
        'Near(Y3-4)': round(pv_near, 1),
        'Far(Y5)': round(pv_far, 1),
        '可行': '是' if npv_s > 0 else '否'
    })

scenario_df = pd.DataFrame(scenario_results)
print("=== 天道推演：三路径场景分析 ===")
print(scenario_df.to_string(index=False))

# 天道推演风险预警
bull_npv = scenario_results[0]['NPV($K)']
bear_npv = scenario_results[2]['NPV($K)']
print(f"\n=== 天道推演风险预警 ===")
print(f"乐观-悲观跨度: ${bull_npv - bear_npv:.1f}K (不确定性范围)")
print(f"悲观场景NPV: ${bear_npv:.1f}K {'(正, 可承受)' if bear_npv > 0 else '(负, 不可承受)'}")
print(f"关键风险: 推理成本上升 + 客户增长放缓 + ARPU下降三重打击")
print(f"缓解策略: (1)多模型策略降低推理成本 (2)outcome-based pricing锁定ARPU (3)渠道多元化分散增长风险")


## 6. 反思与前沿

### 反思问题
1. MarketingAgent Pro 的 NPV 是多少？IRR 是否高于折现率？投资可行吗？
2. 蒙特卡洛模拟的 P(NPV>0) 是多少？5% 和 95% 分位差距说明了什么？
3. 敏感性分析中哪个参数对 NPV 影响最大？推理成本（通过毛利率）排第几？
4. 天道推演的三场景中，Bear case 的 NPV 是多少？风险预警是什么？
5. 如果 DeepSeek 将推理成本降低 90%，毛利率提升后 NPV 如何变化？

### 2026前沿：贝叶斯估值（Bayesian Valuation）
传统 DCF 给出点估计 NPV，蒙特卡洛给出频率派分布。**贝叶斯估值**用 PyMC 构建参数的
后验分布，结合先验信息和观测数据，给出更稳健的估值后验分布。

### Day 1-5 整合（技能4收官）
| Day | 能力 | Day 5 整合角色 |
|-----|------|--------------|
| Day 1 | AI 商业模式类型学 | 画布的客户细分 + 价值主张 |
| Day 2 | AI 定价策略 | 画布的收入流（outcome-based） |
| Day 3 | Agent 经济学 | 画布的成本结构（推理成本） |
| Day 4 | 平台生态战略 | 画布的核心伙伴 + 渠道 |
| Day 5 | 商业模式画布 + 投资评估 | **整合为完整投资评估** |
